# [센서 실습 24] 나침반(자력계) 센서 – QMC5883L (GY-271)

3축 자력계 **QMC5883L**로 지구 자기장을 측정해 **방위각**을 계산합니다. 주변 금속 때문에 생기는 **하드 아이언 오차**를 회전 측정으로 보정하는 방법도 실습합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
지구는 거대한 자석이며, 자력계는 이 **지구 자기장(한국 수평 성분 약 30µT)** 의 방향을 X·Y·Z 세 축으로 측정합니다.
- 센서를 수평으로 두면 **방위각 = atan2(Y, X)** (자북 기준)
- 진북과 자북의 차이 = **자기 편각**: 한국은 서쪽으로 약 8~9° → 진북 방위 = 자북 방위 − 8.5°
- **하드 아이언 오차**: 기판의 나사·스피커 같은 자성체 때문에 측정값 전체가 한쪽으로 **밀림**
  → 센서를 한 바퀴 돌리면 원래 원점 중심 원이어야 할 X–Y 그림이 **중심이 옮겨진 원**이 됨
  → 회전하며 측정한 최댓값·최솟값의 **가운데를 빼면** 보정됨
- 데이터는 레지스터 `0x00`부터 6바이트, **하위 바이트가 먼저** 오는 리틀 엔디언(MPU-6050과 반대)

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 범위 | ±2 / ±8 가우스 |
| 분해능 | 16비트 (±8G: 3000 LSB/G) |
| 방위 정확도 | 1 ~ 2° (보정 후) |
| 전원 | 3.3V |
| 통신 | I2C 주소 0x0D |

## 2. 라즈베리파이 연결 방법
**준비물**: GY-271 모듈(QMC5883L 칩), 점퍼선 4개

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| QMC5883L VCC | 3.3V (1번 핀) |  |
| QMC5883L GND | GND (6번 핀) |  |
| QMC5883L SCL | GPIO3 / SCL (5번 핀) | I2C 클록 |
| QMC5883L SDA | GPIO2 / SDA (3번 핀) | I2C 데이터 (주소 0x0D) |

> 💡 'HMC5883L'로 판매되는 GY-271 모듈도 요즘은 대부분 **QMC5883L** 칩입니다. `i2cdetect -y 1`에서 0x0D가 보이면 QMC, 0x1E면 HMC입니다.

**인터페이스 활성화**
- **I2C**: `sudo raspi-config` → Interface Options → **I2C** → Enable, 연결 확인: `i2cdetect -y 1`

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install smbus2
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
리틀 엔디언 6바이트 → 자기장(G) → 방위각(°) 변환과 방위 이름 함수입니다.

In [ ]:
SCALE = 3000.0          # ±8G 범위: 3000 LSB/G
DECLINATION = -8.5      # 한국 자기 편각 (서편 약 8.5°)

def to_int16_le(lo, hi):
    v = (hi << 8) | lo
    return v - 65536 if v >= 32768 else v

def bytes_to_mag(d):
    """[XL, XH, YL, YH, ZL, ZH] → (mx, my, mz) 가우스"""
    return tuple(round(to_int16_le(d[i], d[i + 1]) / SCALE, 4) for i in (0, 2, 4))

def heading_deg(mx, my, offset=(0.0, 0.0)):
    h = np.degrees(np.arctan2(my - offset[1], mx - offset[0])) + DECLINATION
    return round(h % 360, 1)

def direction_name(h):
    names = ["북", "북동", "동", "남동", "남", "남서", "서", "북서"]
    return names[int((h + 22.5) % 360 // 45)]

### 4-2. 실제 센서 클래스
연속 측정 모드(`0x09` ← `0x1D`)로 설정하고 6바이트를 읽습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealQMC5883L:
    ADDR, DATA, CONTROL, SET_RESET = 0x0D, 0x00, 0x09, 0x0B

    def __init__(self, bus=1):
        from smbus2 import SMBus
        self.bus = SMBus(bus)
        self.bus.write_byte_data(self.ADDR, self.SET_RESET, 0x01)
        self.bus.write_byte_data(self.ADDR, self.CONTROL, 0x1D)   # 연속 측정, 200Hz, ±8G, OSR 512

    def read(self):
        d = self.bus.read_i2c_block_data(self.ADDR, self.DATA, 6)
        mx, my, mz = bytes_to_mag(d)
        return {"mx": mx, "my": my, "mz": mz}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 동작 | 수평으로 들고 10초에 한 바퀴씩, 2바퀴 회전 | 보정용 회전 |
| 지구 자기장 | 수평 0.30 G, 수직 0.40 G | 한국 지역 지자기 세기 |
| 하드 아이언 오차 | X +0.12 G, Y −0.08 G 밀림 | 근처 금속의 영향 |
| 잡음 | ±0.004 G | |

가상 센서는 방향에 따른 자기장을 계산해 **리틀 엔디언 6바이트**로 바꿔 넘깁니다.

In [ ]:
class VirtualQMC5883L:
    OFFSET = (0.12, -0.08)                                  # 하드 아이언 오차 (G)

    def __init__(self, dt=0.1, seed=24):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def read(self):
        true_heading = (360 * self.t / 10) % 360            # 10초에 한 바퀴
        mag_heading = np.radians(true_heading - DECLINATION)
        mx = 0.30 * np.cos(mag_heading) + self.OFFSET[0] + self.rng.normal(0, 0.004)
        my = 0.30 * np.sin(mag_heading) + self.OFFSET[1] + self.rng.normal(0, 0.004)
        mz = 0.40 + self.rng.normal(0, 0.004)
        d = []
        for v in (mx, my, mz):
            r = int(round(v * SCALE)) & 0xFFFF
            d += [r & 0xFF, r >> 8]                         # 하위 바이트 먼저
        self.t += self.dt
        mx, my, mz = bytes_to_mag(d)
        return {"mx": mx, "my": my, "mz": mz, "true_heading": round(true_heading, 1)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualQMC5883L(dt=0.1)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
0.1초 간격으로 20초(두 바퀴) 동안 측정합니다. (가상 센서는 비교용으로 실제 방위 `true_heading`도 함께 기록합니다.)

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 200               # 측정 횟수
INTERVAL = 0.1               # 측정 간격(초)

sensor = RealQMC5883L() if USE_REAL_SENSOR else VirtualQMC5883L(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. 회전 중 X·Y 최댓값·최솟값의 가운데를 **하드 아이언 오프셋**으로 구합니다.
2. 보정 전·후 방위각을 계산하고 실제 방위와의 오차를 비교합니다.

In [ ]:
off = ((df.mx.max() + df.mx.min()) / 2, (df.my.max() + df.my.min()) / 2)
print(f"추정한 하드 아이언 오프셋: X {off[0]:+.3f} G, Y {off[1]:+.3f} G")
df["heading_raw"] = [heading_deg(x, y) for x, y in zip(df.mx, df.my)]
df["heading_cal"] = [heading_deg(x, y, off) for x, y in zip(df.mx, df.my)]
if "true_heading" in df:
    err = lambda col: ((df[col] - df.true_heading + 180) % 360 - 180).abs()
    print(f"방위 오차 (평균/최대): 보정 전 {err('heading_raw').mean():.1f}° / {err('heading_raw').max():.1f}°,"
          f"  보정 후 {err('heading_cal').mean():.1f}° / {err('heading_cal').max():.1f}°")
for i in range(0, 100, 12):
    r = df.iloc[i]
    print(f"  {r.time_s:4.1f}초  방위 {r.heading_cal:5.1f}° ({direction_name(r.heading_cal)})")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].plot(df.mx, df.my, ".", color="gray", ms=4, label="보정 전")
ax[0].plot(df.mx - off[0], df.my - off[1], "k.", ms=4, label="보정 후")
ax[0].axhline(0, color="gray", lw=0.5); ax[0].axvline(0, color="gray", lw=0.5)
ax[0].set_aspect("equal"); ax[0].set_xlabel("X (G)"); ax[0].set_ylabel("Y (G)"); ax[0].legend()
ax[0].set_title("X–Y 자기장 (한 바퀴 = 원)")
ax[1].plot(df.time_s, df.heading_raw, ".", color="gray", ms=3, label="보정 전")
ax[1].plot(df.time_s, df.heading_cal, "k.", ms=3, label="보정 후")
ax[1].set_yticks(range(0, 361, 90)); ax[1].set_yticklabels(["북 0", "동 90", "남 180", "서 270", "북 360"])
ax[1].set_xlabel("시간 (초)"); ax[1].set_title("방위각"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "나침판센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "QMC5883L (GY-271)",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/compass", payload)
# client.disconnect()

## 직접 해 보기
- 하드 아이언 오프셋을 (0.25, 0.20)으로 키우면 보정 전 오차는 얼마나 커지나요?
- 센서를 기울이면 방위가 틀어지는 이유를 생각해 보세요. (기울기 보정: 18번 가속도 센서 활용)
- 방위를 화살표로 표시하는 간단한 나침반 그림을 그려 보세요.